In [10]:
import numpy as np
import pandas as pd

# Display all columns and avoid truncated output
pd.set_option("display.max_columns", None)
print("Libraries imported successfully!")

Libraries imported successfully!


In [11]:
# Load the raw dataset
file_path = r"C:\Users\Owner\Documents\Data_Cleaning_Portfolio\Healthcare-Diabetes.csv"
df = pd.read_csv(file_path)

print(f"Dataset successfully loaded!")
print(f"Total Rows: {df.shape[0]} | Total Columns: {df.shape[1]}")

Dataset successfully loaded!
Total Rows: 2768 | Total Columns: 10


In [12]:
# View data types and non-null counts
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2768 entries, 0 to 2767
Data columns (total 10 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Id                        2768 non-null   int64  
 1   Pregnancies               2768 non-null   int64  
 2   Glucose                   2768 non-null   int64  
 3   BloodPressure             2768 non-null   int64  
 4   SkinThickness             2768 non-null   int64  
 5   Insulin                   2768 non-null   int64  
 6   BMI                       2768 non-null   float64
 7   DiabetesPedigreeFunction  2768 non-null   float64
 8   Age                       2768 non-null   int64  
 9   Outcome                   2768 non-null   int64  
dtypes: float64(2), int64(8)
memory usage: 216.4 KB


In [13]:
# 1. Check for duplicate rows (excluding the unique Id column)
duplicate_count = df.duplicated(subset=df.columns.difference(['Id'])).sum()
print(f"Duplicate Rows (excluding Id): {duplicate_count}")

# 2. Check for biologically impossible zero values across clinical features
zero_counts = (df == 0).sum()
print("\n--- Count of Zero Values per Column ---")
print(zero_counts)

# 3. View summary statistics for numerical columns
df.describe().T[['min', 'mean', 'max']]

Duplicate Rows (excluding Id): 1990

--- Count of Zero Values per Column ---
Id                             0
Pregnancies                  412
Glucose                       18
BloodPressure                125
SkinThickness                800
Insulin                     1330
BMI                           39
DiabetesPedigreeFunction       0
Age                            0
Outcome                     1816
dtype: int64


,min,mean,max
Id,1.000,1384.500000,2768.00
Pregnancies,0.000,3.742775,17.00
Glucose,0.000,121.102601,199.00
BloodPressure,0.000,69.134393,122.00
SkinThickness,0.000,20.824422,110.00
Insulin,0.000,80.127890,846.00
BMI,0.000,32.137392,80.60
DiabetesPedigreeFunction,0.078,0.471193,2.42
Age,21.000,33.132225,81.00
Outcome,0.000,0.343931,1.00


In [14]:
# 1. Clean column names (lowercase with underscores)
df.columns = df.columns.str.strip().str.lower()
df = df.rename(
    columns={
        "bloodpressure": "blood_pressure",
        "skinthickness": "skin_thickness",
        "diabetespedigreefunction": "diabetes_pedigree_function",
    }
)

# 2. Drop duplicates (ignoring 'id' column)
feature_cols = [col for col in df.columns if col != "id"]
df_clean = df.drop_duplicates(subset=feature_cols).copy()

# 3. Drop the redundant 'id' column
df_clean = df_clean.drop(columns=["id"])

print(f"Original shape: {df.shape}")
print(f"Cleaned shape after dropping duplicates: {df_clean.shape}")
df_clean.head()

Original shape: (2768, 10)
Cleaned shape after dropping duplicates: (778, 9)


,pregnancies,glucose,blood_pressure,skin_thickness,insulin,bmi,diabetes_pedigree_function,age,outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [15]:
# Columns where 0 is biologically impossible
zero_invalid_cols = [
    "glucose",
    "blood_pressure",
    "skin_thickness",
    "insulin",
    "bmi",
]

# Replace 0 with NaN for these specific columns
df_clean[zero_invalid_cols] = df_clean[zero_invalid_cols].replace(0, np.nan)

# Check missing value percentage per column
missing_summary = pd.DataFrame(
    {
        "Missing_Count": df_clean[zero_invalid_cols].isnull().sum(),
        "Missing_Percentage": (
            df_clean[zero_invalid_cols].isnull().mean() * 100
        ).round(2),
    }
)

print("--- Hidden Missing Values Converted to NaN ---")
print(missing_summary)

--- Hidden Missing Values Converted to NaN ---
                Missing_Count  Missing_Percentage
glucose                     5                0.64
blood_pressure             36                4.63
skin_thickness            229               29.43
insulin                   377               48.46
bmi                        11                1.41


In [16]:
# Calculate median values grouped by Outcome (0 = Non-Diabetic, 1 = Diabetic)
print("--- Baseline Medians by Outcome Group ---")
print(df_clean.groupby("outcome")[zero_invalid_cols].median().T)

# Fill missing values using the group median based on 'outcome'
for col in zero_invalid_cols:
    df_clean[col] = df_clean[col].fillna(
        df_clean.groupby("outcome")[col].transform("median")
    )

# Verify that 0 missing values remain across all columns
print("\n--- Remaining Missing Values ---")
print(df_clean.isnull().sum())

--- Baseline Medians by Outcome Group ---
outcome             0      1
glucose         108.0  140.0
blood_pressure   70.0   74.5
skin_thickness   27.0   32.5
insulin         105.0  173.0
bmi              30.4   34.3

--- Remaining Missing Values ---
pregnancies                   0
glucose                       0
blood_pressure                0
skin_thickness                0
insulin                       0
bmi                           0
diabetes_pedigree_function    0
age                           0
outcome                       0
dtype: int64


In [17]:
# 1. Create a categorical BMI risk group feature
bmi_bins = [0, 18.5, 24.9, 29.9, 100]
bmi_labels = ["Underweight", "Normal", "Overweight", "Obese"]
df_clean["bmi_category"] = pd.cut(
    df_clean["bmi"], bins=bmi_bins, labels=bmi_labels
)

# 2. Create an Age Group feature
age_bins = [20, 30, 40, 50, 100]
age_labels = ["21-30", "31-40", "41-50", "50+"]
df_clean["age_group"] = pd.cut(
    df_clean["age"], bins=age_bins, labels=age_labels
)

# Preview the engineered features
df_clean[["age", "age_group", "bmi", "bmi_category", "outcome"]].head()

,age,age_group,bmi,bmi_category,outcome
0,50,41-50,33.6,Obese,1
1,31,31-40,26.6,Overweight,0
2,32,31-40,23.3,Normal,1
3,21,21-30,28.1,Overweight,0
4,33,31-40,43.1,Obese,1


In [18]:
# 1. Cast integer columns back to int64 (skin_thickness and blood_pressure)
int_cols = ["pregnancies", "glucose", "blood_pressure", "skin_thickness", "age", "outcome"]
df_clean[int_cols] = df_clean[int_cols].astype(int)

# 2. Save the final cleaned dataset to your portfolio directory
output_path = r"C:\Users\Owner\Documents\Data_Cleaning_Portfolio\cleaned_healthcare_diabetes.csv"
df_clean.to_csv(output_path, index=False)

print(f"SUCCESS! Cleaned dataset saved to:\n{output_path}\n")

# 3. Final summary inspection
print(f"Final Cleaned Dataset Shape: {df_clean.shape[0]} rows, {df_clean.shape[1]} columns")
df_clean.info()

SUCCESS! Cleaned dataset saved to:
C:\Users\Owner\Documents\Data_Cleaning_Portfolio\cleaned_healthcare_diabetes.csv

Final Cleaned Dataset Shape: 778 rows, 11 columns
<class 'pandas.DataFrame'>
Index: 778 entries, 0 to 2336
Data columns (total 11 columns):
 #   Column                      Non-Null Count  Dtype   
---  ------                      --------------  -----   
 0   pregnancies                 778 non-null    int64   
 1   glucose                     778 non-null    int64   
 2   blood_pressure              778 non-null    int64   
 3   skin_thickness              778 non-null    int64   
 4   insulin                     778 non-null    float64 
 5   bmi                         778 non-null    float64 
 6   diabetes_pedigree_function  778 non-null    float64 
 7   age                         778 non-null    int64   
 8   outcome                     778 non-null    int64   
 9   bmi_category                778 non-null    category
 10  age_group                   778 non-null  